
# Supuestos: Preparación de Datos, Exposición y Dispersión

En esta práctica vamos a revisar **los supuestos fundamentales para pricing** antes de correr modelos estadísticos más avanzados.

## Objetivos
1. Comprender la importancia de **unir datos de pólizas y siniestros**.
2. Calcular y utilizar la **exposición** en años-póliza.
3. Analizar la **dispersión de los datos** y entender el concepto de **heterocedasticidad**.
4. Revisar cómo la **segmentación** mejora la homogeneidad de los datos.



## 1. Unión de datos

En la práctica real, los datos de pólizas y siniestros vienen en tablas separadas.  
El primer paso es unirlos de manera correcta utilizando el `id_poliza`.



In [ ]:

import pandas as pd

# Simulamos datos de pólizas y siniestros
polizas = pd.DataFrame({
    "id_poliza": [1,2,3,4],
    "fecha_ini": pd.to_datetime(["2022-01-01","2022-03-01","2022-06-01","2022-09-01"]),
    "fecha_fin": pd.to_datetime(["2022-12-31","2022-08-31","2022-12-31","2022-12-31"]),
    "prima": [12000, 8000, 15000, 10000]
})

siniestros = pd.DataFrame({
    "id_poliza": [1,2,2,4],
    "fecha_siniestro": pd.to_datetime(["2022-05-01","2022-04-15","2022-07-20","2022-11-02"]),
    "monto": [5000, 3000, 7000, 20000]
})

# Unión
df = pd.merge(polizas, siniestros, on="id_poliza", how="left")
df



## 2. Exposición

La exposición mide el tiempo durante el cual la compañía está en riesgo.  
En este ejemplo se calcula como la diferencia entre `fecha_fin` y `fecha_ini`, expresada en **años-póliza**.



In [ ]:

df["exposicion"] = (df["fecha_fin"] - df["fecha_ini"]).dt.days / 365
df[["id_poliza","exposicion"]]



## 3. Dispersión y heterocedasticidad

En seguros, la varianza crece con la media → los datos presentan **heterocedasticidad**.  
Lo visualizamos comparando la media y la varianza de montos de siniestros por póliza.



In [ ]:

import numpy as np

tabla = df.groupby("id_poliza").agg(
    siniestros=("monto","count"),
    monto_total=("monto","sum"),
    exposicion=("exposicion","mean")
).fillna(0)

tabla["frecuencia"] = tabla["siniestros"]/tabla["exposicion"]
tabla["severidad"] = np.where(tabla["siniestros"]>0, tabla["monto_total"]/tabla["siniestros"], 0)

media = tabla["severidad"].mean()
varianza = tabla["severidad"].var()

tabla, media, varianza


In [ ]:

import matplotlib.pyplot as plt

plt.scatter(tabla["frecuencia"], tabla["severidad"])
plt.xlabel("Frecuencia")
plt.ylabel("Severidad")
plt.title("Ejemplo de heterocedasticidad")
plt.show()



## 4. Segmentación

La segmentación busca dividir la cartera en grupos más homogéneos para reducir la dispersión.  
Ejemplo: segmentar por antigüedad de la póliza o por zona geográfica.

